In [1]:
!pip install -q keras-nlp --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.8 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
import numpy as np


/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/config.json
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/tokenizer.json
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/metadata.json
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/model.weights.h5
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/assets/tokenizer/vocabulary.spm
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


In [3]:
os.environ["KERAS_BACKEND"] = "tensorflow"  # 也可以設為 "jax" 或 "torch"


import keras
import keras_nlp
import wandb
from sklearn.model_selection import train_test_split
from wandb.integration.keras import WandbMetricsLogger
# 初始化 W&B 實驗追蹤
wandb.login(key="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")
wandb.init(project="jigsaw-deberta-keras", name="deberta-v3-local-keras-run")
# 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

2026-05-28 11:15:04.477925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779966904.670133      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779966904.725263      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779966905.173775      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779966905.173810      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779966905.173813      23 computation_placer.cc:177] computation placer alr

In [4]:
# 讀取剛剛印出來的 toxic comment 訓練資料 (路徑請根據你執行出來的結果修改)
train_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip')
train_df.head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
test_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip')
test_df.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [6]:
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df['comment_text'].values, 
    train_df[LABELS].values, 
    test_size=0.35, 
    random_state=42
)

In [8]:
# 貼上你在右邊 Input 複製的本地路徑
MODEL_PATH = "/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3"

# 1. 建立文字預處理器（自動讀取本地斷詞權重，長度設為作業要求的 192）
preprocessor = keras_nlp.models.DebertaV3Preprocessor.from_preset(
    MODEL_PATH,
    sequence_length=192 
)

# 2. 建立分類器（自動載入本地模型結構，隨機初始化 6 個輸出的分類頭）
model = keras_nlp.models.DebertaV3Classifier.from_preset(
    MODEL_PATH,
    preprocessor=preprocessor,
    num_classes=len(LABELS),
    activation="sigmoid"  # 多標籤分類核心：各欄位獨立機率
)

I0000 00:00:1779966933.053586      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779966933.059535      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [9]:
# 編譯模型
model.compile(
    loss="binary_crossentropy",  # 多標籤是非題專用的損失函數
    optimizer=keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=0.01),
    metrics=[keras.metrics.AUC(multi_label=True, name="mean_auc")], # Kaggle 官方指標
    jit_compile=True # 開啟 XLA 加速，讓 TensorFlow 幫你的 GPU 進行底層優化
)

In [10]:
# 這裡會花費大約 1 到 1.5 小時，你可以隨時去 W&B 網站上看即時圖表
model.fit(
    x=X_train, 
    y=y_train,
    validation_data=(X_val, y_val), # 每個 Epoch 結束自動考一次模擬考
    batch_size=16, 
    epochs=1,
    callbacks=[WandbMetricsLogger()] # 自動把進度丟給 W&B
)

# 訓練完畢，關閉 W&B 線上紀錄
wandb.finish()


I0000 00:00:1779966991.675247     104 service.cc:152] XLA service 0x7be890009040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779966991.675285     104 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779966991.675289     104 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779966999.360892     104 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779967041.772891     104 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6483/6483 ━━━━━━━━━━━━━━━━━━━━ 7550s 1s/step - loss: 0.0493 - mean_auc: 0.9656 - val_loss: 0.0457 - val_mean_auc: 0.9833


wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:         epoch/epoch ▁
wandb: epoch/learning_rate ▁
wandb:          epoch/loss ▁
wandb:      epoch/mean_auc ▁
wandb:      epoch/val_loss ▁
wandb:  epoch/val_mean_auc ▁
wandb: 
wandb: Run summary:
wandb:         epoch/epoch 0
wandb: epoch/learning_rate 2e-05
wandb:          epoch/loss 0.04934
wandb:      epoch/mean_auc 0.96562
wandb:      epoch/val_loss 0.04574
wandb:  epoch/val_mean_auc 0.98332
wandb: 
wandb: 🚀 View run deberta-v3-local-keras-run at: https://wandb.ai/072599-chingshin-academy/jigsaw-deberta-keras/runs/6rgbq51m
wandb: ⭐️ View project at: https://wandb.ai/072599-chingshin-academy/jigsaw-deberta-keras
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260528_111525-6rgbq51m/logs


In [11]:
print("====== 🏆 訓練完成！立刻對測試集進行預測 (第六區) ======")
test_df = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip")

predictions = model.predict(test_df['comment_text'].values, batch_size=16, verbose=1)

submission_df = pd.DataFrame()
submission_df['id'] = test_df['id']
for i, label in enumerate(LABELS):
    submission_df[label] = predictions[:, i]

submission_df.to_csv("submission_deberta_keras.csv", index=False)
wandb.finish()

print("🎉【大功告成】預測檔案 'submission_deberta_keras.csv' 已成功在右下角 Output 生成！")

====== 🏆 訓練完成！立刻對測試集進行預測 (第六區) ======
9573/9573 ━━━━━━━━━━━━━━━━━━━━ 3148s 328ms/step
🎉【大功告成】預測檔案 'submission_deberta_keras.csv' 已成功在右下角 Output 生成！
